In [5]:
# Implementation3: Constructing a Log Likelihood Estimator inspired by Nguyen 

import numpy as np

In [6]:
class StreamingLLREstimator:
    """
    Estimator of log-likelihood ratio by the f-divergence variational form
    (Nguyen et al., 2009—§2).
    """
    def __init__(self, feature_map, dim, lr=1e-3, l2=1e-4):
        """
        Args:
            feature_map: callable phi(x) -> (d,) feature vector
            dim:         dimension d of the feature map
            lr:          base learning rate for SGD updates
            l2:          weight of L2 regularization on w
        """
        self.phi = feature_map
        self.w   = np.zeros(dim)
        self.lr  = lr
        self.l2  = l2
        # optional running counters, for diagnostics
        self.nP = 0
        self.nQ = 0

    def update_P(self, x):
        """
        Incorporate one sample x ~ P: gradient of E_P[w^T phi(x)] is +phi(x).
        """
        self.nP += 1
        g = self.phi(x)                 # ∇_w [w^T φ(x)] = φ(x)
        # apply weight decay
        g -= self.l2 * self.w
        # update
        self.w += self.lr * g

    def update_Q(self, y):
        """
        Incorporate one sample y ~ Q: gradient of -E_Q[exp(w^T φ(y))] is
        -exp(w^T φ(y)) * φ(y).
        """
        self.nQ += 1
        t = np.dot(self.w, self.phi(y))
        e = np.exp(t)
        g = -e * self.phi(y)            # ∇_w [ -e^{w^T φ(y)} ] = -e^{t} φ(y)
        # apply weight decay
        g -= self.l2 * self.w
        # update
        self.w += self.lr * g

    def predict_llr(self, x):
        """
        Returns the estimated log-likelihood ratio log p(x)/q(x) ≈ w^T φ(x).
        """
        return float(np.dot(self.w, self.phi(x)))

    def state(self):
        """Optional: return diagnostics."""
        return {'w': self.w.copy(), 'nP': self.nP, 'nQ': self.nQ}
